<a href="https://colab.research.google.com/github/HereLiesAz/PaperPlanes/blob/main/PaperPlanes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install rembg torch diffusers transformers accelerate opencv-python-headless pillow numpy

In [25]:
!pip install "rembg[cpu]" --upgrade
import os
import io
import zipfile
import cv2
import torch
import numpy as np
import requests
import re
from PIL import Image
from rembg import remove
from diffusers import StableDiffusionImg2ImgPipeline
from transformers import pipeline as hf_pipeline
from google.colab import drive

# --- MOUNT THE VAULT ---
print("Authenticating with Google Drive...")
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/PaperPlanes_Autopsy"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Output directory secured at: {DRIVE_DIR}")

def vivisect(image_path, layers=6):
    base_name = os.path.basename(image_path).split('.')[0]
    zip_path = os.path.join(DRIVE_DIR, f"{base_name}_strata.zip")

    if os.path.exists(zip_path):
        print(f"Skipping {image_path}: Artifact already exists at {zip_path}")
        return

    print(f"\n--- Vivisecting {image_path} ---")
    print("Phase 1: Stripping reality...")
    orig_pil = Image.open(image_path).convert("RGB")
    nobg_pil = remove(orig_pil)
    subject_mask = np.array(nobg_pil)[:, :, 3] > 0
    orig_cv = cv2.cvtColor(np.array(orig_pil), cv2.COLOR_RGB2BGR)

    print("Phase 2: Forging the hallucination...")
    torch.cuda.empty_cache()
    sd_input_image = orig_pil.resize((512, 512), Image.LANCZOS)
    sd_pipe = StableDiffusionImg2ImgPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
    sd_pipe.safety_checker = None
    prompt = "A high fidelity 3D render, volumetric depth, clear geometry, structural, high contrast"
    gen_pil = sd_pipe(prompt=prompt, image=sd_input_image, strength=0.65, guidance_scale=7.5).images[0]
    del sd_pipe
    del sd_input_image
    torch.cuda.empty_cache()

    print("Phase 3: Bruteforce alignment...")
    gen_cv = cv2.cvtColor(np.array(gen_pil), cv2.COLOR_RGB2BGR)
    gen_cv = cv2.resize(gen_cv, (orig_cv.shape[1], orig_cv.shape[0]))
    gray_src = cv2.cvtColor(orig_cv, cv2.COLOR_BGR2GRAY)
    gray_tgt = cv2.cvtColor(gen_cv, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(nfeatures=5000)
    kp_src, des_src = orb.detectAndCompute(gray_src, None)
    kp_tgt, des_tgt = orb.detectAndCompute(gray_tgt, None)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    matches = bf.knnMatch(des_src, des_tgt, k=2)
    good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]

    if len(good_matches) > 10:
        src_pts = np.float32([kp_src[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        tgt_pts = np.float32([kp_tgt[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        matrix, _ = cv2.findHomography(tgt_pts, src_pts, cv2.RANSAC, 5.0)
        aligned_cv = cv2.warpPerspective(gen_cv, matrix, (orig_cv.shape[1], orig_cv.shape[0])) if matrix is not None else gen_cv
    else:
        aligned_cv = gen_cv

    aligned_pil = Image.fromarray(cv2.cvtColor(aligned_cv, cv2.COLOR_BGR2RGB))

    print("Phase 4: Extracting depth map...")
    depth_pipe = hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)
    depth_array = np.array(depth_pipe(aligned_pil)["depth"]).astype(np.float32)
    del depth_pipe
    torch.cuda.empty_cache()

    print("Phase 5: Slicing strata...")
    subject_depth = depth_array[subject_mask]
    min_d, max_d = subject_depth.min(), subject_depth.max()
    normalized_depth = np.zeros_like(depth_array)
    if min_d != max_d:
        normalized_depth[subject_mask] = np.interp(depth_array[subject_mask], (min_d, max_d), (0, 255))

    bins = np.linspace(0, 255.1, layers + 1)
    orig_array = np.array(orig_pil)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for i in range(layers):
            layer_mask = (normalized_depth >= bins[i]) & (normalized_depth < bins[i+1]) & subject_mask
            if np.any(layer_mask):
                layer_rgba = np.zeros((orig_array.shape[0], orig_array.shape[1], 4), dtype=np.uint8)
                layer_rgba[..., :3] = orig_array
                layer_rgba[..., 3] = cv2.GaussianBlur((layer_mask * 255).astype(np.uint8), (5, 5), 0)
                img_byte_arr = io.BytesIO()
                Image.fromarray(layer_rgba).save(img_byte_arr, format='PNG')
                zf.writestr(f"layer_{i:03d}.png", img_byte_arr.getvalue())

    print(f"Vivisection complete. Artifact permanently sealed in Drive: {zip_path}")

def raid_album(shared_url):
    print(f"Raiding the void: {shared_url}")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(shared_url, headers=headers)
    matches = re.findall(r'(https:\/\/lh3\.googleusercontent\.com\/[a-zA-Z0-9\-_]+)', response.text)
    unique_urls = list(set(matches))
    image_urls = [url for url in unique_urls if len(url) > 60]

    if not image_urls:
        print("The mass grave is empty or fortified.")
        return []

    os.makedirs("victims", exist_ok=True)
    corpses = []
    for i, url in enumerate(image_urls):
        try:
            img_data = requests.get(url + "=w2048-h2048", headers=headers).content
            filename = f"victims/victim_{i:03d}.jpg"
            with open(filename, "wb") as f:
                f.write(img_data)
            corpses.append(filename)
            print(f"Exhumed: {filename}")
        except Exception as e:
            print(f"Failed to exhume {url}: {e}")
    return corpses

ALBUM_URL = "https://photos.app.goo.gl/dguJYHjN1ZrmwfFU6"

# --- UPDATED EXECUTION LOGIC ---
targets = []

# Attempt scraping if URL provided
if ALBUM_URL and ALBUM_URL.startswith("https"):
    targets.extend(raid_album(ALBUM_URL))

# Always check local directory as well
local_files = [f for f in os.listdir('.') if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')) and not f.startswith('.')]
if local_files:
    print(f"Detected {len(local_files)} local victims in the chamber.")
    targets.extend(local_files)

if targets:
    # Use unique list to prevent double processing
    for victim in list(set(targets)):
        vivisect(victim, layers=6)
else:
    print("No victims found in the void or local chamber.")

Authenticating with Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directory secured at: /content/drive/MyDrive/PaperPlanes_Autopsy
Raiding the void: https://photos.app.goo.gl/dguJYHjN1ZrmwfFU6
The mass grave is empty or fortified.
Detected 194 local victims in the chamber.
Skipping FB_IMG_1547240926532.jpg: Artifact already exists at /content/drive/MyDrive/PaperPlanes_Autopsy/FB_IMG_1547240926532_strata.zip
Skipping original_5a64157e-6338-4c63-9916-e9695d92d3fc_PXL_20221201_163506286.jpg: Artifact already exists at /content/drive/MyDrive/PaperPlanes_Autopsy/original_5a64157e-6338-4c63-9916-e9695d92d3fc_PXL_20221201_163506286_strata.zip
Skipping 20230704_002955~2.jpg: Artifact already exists at /content/drive/MyDrive/PaperPlanes_Autopsy/20230704_002955~2_strata.zip
Skipping bridge.jpg: Artifact already exists at /content/drive/MyDrive/PaperPlanes_Autopsy/bridge_strata.zip
Skipping I

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

Phase 3: Bruteforce alignment...
Phase 4: Extracting depth map...


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Phase 5: Slicing strata...


IndexError: boolean index did not match indexed array along axis 0; size of axis is 3120 but size of corresponding boolean axis is 4160

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive
